<a href="https://colab.research.google.com/github/ankitaguptacs24-ai/BIS/blob/main/LAB1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import random
import numpy as np

# ==========================================
# 1. DATA STRUCTURES & BENCHMARK INITIALIZATION
# ==========================================
class Customer:
    def __init__(self, node_id, x, y, demand, ready_time, due_time, service_time):
        self.id = node_id
        self.x = x
        self.y = y
        self.demand = demand
        self.ready_time = ready_time
        self.due_time = due_time
        self.service_time = service_time

def euclidean_distance(c1, c2):
    return math.hypot(c1.x - c2.x, c1.y - c2.y)

# Sample VRPTW Instance Data (Simplified Solomon format)
VEHICLE_CAPACITY = 200
PENALTY_WEIGHT = 1000.0  # Gamma weight for time window violations

depot = Customer(0, 40.0, 50.0, 0, 0, 1236, 0)
customers = [
    Customer(1, 45.0, 68.0, 10, 912, 967, 90),
    Customer(2, 45.0, 70.0, 30, 825, 870, 90),
    Customer(3, 42.0, 66.0, 10, 65, 146, 90),
    Customer(4, 42.0, 68.0, 20, 727, 782, 90),
    Customer(5, 42.0, 65.0, 10, 15, 67, 90),
    Customer(6, 40.0, 69.0, 10, 621, 702, 90),
    Customer(7, 40.0, 66.0, 20, 170, 225, 90),
    Customer(8, 38.0, 68.0, 20, 255, 324, 90),
    Customer(9, 38.0, 70.0, 10, 534, 605, 90),
    Customer(10, 35.0, 66.0, 10, 357, 410, 90)
]
all_nodes = {c.id: c for c in [depot] + customers}

# ==========================================
# 2. SOLOMON INSERTION HEURISTIC (SIH) SEED
# ==========================================
def solomon_insertion_heuristic(customer_list, depot_node, capacity):
    """Generates an initial seed sequence using Solomon Insertion principles."""
    unrouted = set(c.id for c in customer_list)
    chromosome_seed = []

    while unrouted:
        # Start a new seed customer (farthest unrouted customer from depot)
        seed = max(unrouted, key=lambda c_id: euclidean_distance(depot_node, all_nodes[c_id]))
        chromosome_seed.append(seed)
        unrouted.remove(seed)

        # Sequentially insert nearest remaining feasible customers
        current_node = all_nodes[seed]
        while unrouted:
            next_node_id = min(unrouted, key=lambda c_id: euclidean_distance(current_node, all_nodes[c_id]))
            chromosome_seed.append(next_node_id)
            unrouted.remove(next_node_id)
            current_node = all_nodes[next_node_id]

    return chromosome_seed

# ==========================================
# 3. CHROMOSOME DECODER & FITNESS EVALUATION
# ==========================================
def decode_chromosome_to_routes(chromosome):
    """Converts chromosome integer array into feasible vehicle sub-routes."""
    routes = []
    current_route = []
    current_load = 0
    current_time = 0.0
    prev_node = depot

    for cust_id in chromosome:
        cust = all_nodes[cust_id]
        travel_time = euclidean_distance(prev_node, cust)
        arrival_time = current_time + travel_time
        start_service = max(arrival_time, cust.ready_time)

        # Check capacity and due-time constraints
        if (current_load + cust.demand <= VEHICLE_CAPACITY) and (start_service <= cust.due_time):
            current_route.append(cust_id)
            current_load += cust.demand
            current_time = start_service + cust.service_time
            prev_node = cust
        else:
            # Finalize current route (return to depot) and start new vehicle route
            if current_route:
                routes.append(current_route)
            current_route = [cust_id]
            current_load = cust.demand
            arrival_time = euclidean_distance(depot, cust)
            current_time = max(arrival_time, cust.ready_time) + cust.service_time
            prev_node = cust

    if current_route:
        routes.append(current_route)
    return routes

def evaluate_fitness(chromosome):
    """Calculates fitness: Total Travel Time + Penalty Factor for TW violations."""
    routes = decode_chromosome_to_routes(chromosome)
    total_time = 0.0
    total_penalties = 0.0

    for route in routes:
        current_time = 0.0
        prev_node = depot
        for cust_id in route:
            cust = all_nodes[cust_id]
            travel_time = euclidean_distance(prev_node, cust)
            arrival_time = current_time + travel_time

            # Late arrival penalty calculation
            if arrival_time > cust.due_time:
                total_penalties += (arrival_time - cust.due_time)

            start_service = max(arrival_time, cust.ready_time)
            current_time = start_service + cust.service_time
            prev_node = cust

        # Return journey to depot
        total_time += current_time + euclidean_distance(prev_node, depot)

    fitness = total_time + (PENALTY_WEIGHT * total_penalties)
    return fitness, len(routes)

# ==========================================
# 4. GA OPERATORS (CROSSOVER & MUTATION)
# ==========================================
def pmx_crossover(parent1, parent2):
    """Partially Mapped Crossover (PMX) operator."""
    size = len(parent1)
    p1, p2 = np.array(parent1), np.array(parent2)
    cx_point1, cx_point2 = sorted(random.sample(range(size), 2))

    offspring1 = np.full(size, -1)
    offspring1[cx_point1:cx_point2] = p1[cx_point1:cx_point2]

    for i in range(cx_point1, cx_point2):
        if p2[i] not in offspring1:
            val = p2[i]
            pos = i
            while cx_point1 <= pos < cx_point2:
                pos = np.where(p2 == p1[pos])[0][0]
            offspring1[pos] = val

    offspring1[offspring1 == -1] = [item for item in p2 if item not in offspring1]
    return list(offspring1)

def inversion_mutation(chromosome):
    """Inversion Mutation Operator."""
    mutated = chromosome.copy()
    idx1, idx2 = sorted(random.sample(range(len(chromosome)), 2))
    mutated[idx1:idx2] = reversed(mutated[idx1:idx2])
    return mutated

# ==========================================
# 5. MAIN HGA-SIH EXECUTION LOOP
# ==========================================
def run_hga_sih(pop_size=200, generations=100, cx_rate=0.8, mut_rate=0.01):
    customer_ids = [c.id for c in customers]

    # Initialize Population (Hybrid: 10% SIH Seeds + 90% Stochastic Random)
    sih_seed = solomon_insertion_heuristic(customers, depot, VEHICLE_CAPACITY)
    num_sih = int(pop_size * 0.10)

    population = [sih_seed.copy() for _ in range(num_sih)]
    for _ in range(pop_size - num_sih):
        rand_chrom = customer_ids.copy()
        random.shuffle(rand_chrom)
        population.append(rand_chrom)

    best_solution = None
    best_fitness = float('inf')
    best_nv = 0

    for gen in range(generations):
        # Fitness Evaluation
        pop_fitness = [evaluate_fitness(chrom) for chrom in population]

        # Track Global Best
        for i, (fit, nv) in enumerate(pop_fitness):
            if fit < best_fitness:
                best_fitness = fit
                best_nv = nv
                best_solution = population[i]

        # Selection (10% Elitism + 90% Roulette Wheel)
        sorted_pop = [chrom for _, chrom in sorted(zip([f[0] for f in pop_fitness], population))]
        next_gen = sorted_pop[:int(pop_size * 0.10)]  # Elitism

        # Roulette wheel selection pool
        fits = np.array([1.0 / (f[0] + 1e-6) for f in pop_fitness])
        probs = fits / np.sum(fits)

        while len(next_gen) < pop_size:
            p1_idx, p2_idx = np.random.choice(len(population), size=2, p=probs)
            p1, p2 = population[p1_idx], population[p2_idx]

            # Crossover
            if random.random() < cx_rate:
                child = pmx_crossover(p1, p2)
            else:
                child = p1.copy()

            # Mutation
            if random.random() < mut_rate:
                child = inversion_mutation(child)

            next_gen.append(child)

        population = next_gen

    return best_solution, best_fitness, best_nv

# ==========================================
# 6. RUN AND PRINT RESULTS
# ==========================================
if __name__ == "__main__":
    best_chrom, min_cost, num_vehicles = run_hga_sih(pop_size=200, generations=50)
    optimal_routes = decode_chromosome_to_routes(best_chrom)

    print("==================================================")
    print("       HGA-SIH VRPTW OPTIMIZATION RESULTS        ")
    print("==================================================")
    print(f"Optimal Fitness Score (Distance + Penalty) : {min_cost:.2f}")
    print(f"Optimal Fleet Size (Number of Vehicles)    : {num_vehicles}")
    print("\nGenerated Vehicle Sub-Routes:")
    for idx, route in enumerate(optimal_routes, 1):
        print(f" Vehicle {idx}: Depot(0) -> {' -> '.join(map(str, route))} -> Depot(0)")

       HGA-SIH VRPTW OPTIMIZATION RESULTS        
Optimal Fitness Score (Distance + Penalty) : 1329.81
Optimal Fleet Size (Number of Vehicles)    : 2

Generated Vehicle Sub-Routes:
 Vehicle 1: Depot(0) -> 8 -> 10 -> 9 -> 6 -> 4 -> 2 -> 1 -> Depot(0)
 Vehicle 2: Depot(0) -> 5 -> 3 -> 7 -> Depot(0)
